# 13 Command center

Send operating commands to the turbines of Northwind Ridge by hand, one line each, and see each one take effect. For testing the command path, and for showing it.

- Run the first two code cells once per session. After that, run any cell below as often as you like: each is one command.
- The commands travel the path of Phase 5a: the eventstream `WindFarmCommands`, the data flow of Azure IoT Operations, the control topic on the edge. Before the first run: `bash infra/08-create-command-path.sh` on the workstation.
- After a command the notebook waits until the telemetry in `WindFarmEH` shows it, and says how many seconds that took.
- The control topic takes five commands: `curtail`, `release`, `stop`, `start`, `reset`. It cannot inject a fault or change the wind; those stay on the edge (`simctl`).

This notebook is for you. `12_send_edge_command` is the one the operations agent runs as its action: leave the values in that notebook alone, and send by hand from here.

In [ ]:
# Generated by tools/build-phase5.py from tools/edge_command.py and tools/command_center.py. Change it there.
import json
import re
import uuid
from datetime import datetime, timezone

COMMANDS = ("curtail", "release", "stop", "start", "reset")
TURBINE = re.compile(r"^WT-(0[1-9]|1[0-2])$")
DURATION = re.compile(r"^[1-9][0-9]{0,3}[smh]$")
# How a person or an agent may write a duration: 10m, 10 m, 10 min, 10 minutes, 90 seconds, 1 hour.
SPELLED_DURATION = re.compile(r"^([1-9][0-9]{0,3})\s*(s|sec|secs|second|seconds|m|min|mins|minute|minutes|h|hr|hrs|hour|hours)$")
STOP_REASONS = ("operator", "maintenance")


class CommandError(ValueError):
    """The command would not be accepted, or would do more than was asked for."""


def build_command(command, turbine, percent=None, duration="", reason="", requested_by="", request_id=None, now=None):
    """The JSON object for the control topic, after checking every field.

    The simulator treats a missing turbine as "all twelve". Here a turbine is always required, and
    all of them only when the word `all` is written out.
    """
    name = str(command or "").strip().lower()
    if name not in COMMANDS:
        raise CommandError(f"'{command}' is not a command of the control topic. Use one of: {', '.join(COMMANDS)}.")
    target = str(turbine or "").strip()
    if target.lower() == "all":
        target = "all"
    else:
        target = target.upper()
        if not TURBINE.match(target):
            raise CommandError(f"'{turbine}' is not a turbine. Use WT-01 to WT-12, or the word all.")
    out = {"command": name, "requestId": request_id or uuid.uuid4().hex[:12], "turbine": target}

    text = str(duration or "").strip().lower()
    if text and name not in ("curtail", "stop"):
        raise CommandError(f"'{name}' takes no duration.")
    if text and not DURATION.match(text):
        raise CommandError(f"'{duration}' is not a duration. Write a number and s, m or h, for example 30m.")
    if name == "curtail":
        try:
            value = float(percent)
        except (TypeError, ValueError):
            raise CommandError("curtail needs percent: the share of rated power the turbine may still produce, 0 to 100.") from None
        if not 0 <= value <= 100:
            raise CommandError(f"percent is {value:g}; it must be between 0 and 100.")
        if not text:
            raise CommandError("curtail needs a duration here, for example 30m, so that a forgotten command ends by itself. To end it sooner, send release.")
        out["percent"] = value
    elif percent not in (None, ""):
        raise CommandError(f"'{name}' takes no percent.")
    if text:
        out["duration"] = text
    if name == "stop":
        why = str(reason or "operator").strip().lower()
        if why not in STOP_REASONS:
            raise CommandError(f"The reason of a stop is one of: {', '.join(STOP_REASONS)}.")
        out["reason"] = why
    elif reason:
        out["note"] = str(reason)[:200]
    if requested_by:
        out["requestedBy"] = str(requested_by)[:120]
    out["issuedAt"] = (now or datetime.now(timezone.utc)).strftime("%Y-%m-%dT%H:%M:%SZ")
    return out


def as_flag(value, name="dry_run"):
    """True or False from a parameter that may arrive as text. The text "False" is not a true value."""
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in ("true", "1", "yes"):
        return True
    if text in ("false", "0", "no", ""):
        return False
    raise CommandError(f"{name} is '{value}'; write True or False.")


def from_parameters(command, turbine, percent=None, duration="", reason="", requested_by="", request_id=None, now=None):
    """The command for the values of the notebook's parameter cell, however they were written.

    Whoever fills that cell, a person, a pipeline or the operations agent, changes the values the
    command needs and leaves the others as they stand, and may write `Curtail`, `30%` or `10 minutes`.
    So the name is brought to lower case before anything is decided by it, a value the command does
    not take is left out, and a percentage or a duration is read in the forms people write them.
    Deciding by the name as it was typed dropped the duration of a `Stop`, which then had no end.
    """
    name = str(command or "").strip().lower()
    if name == "curtail" and isinstance(percent, str):
        percent = percent.strip().rstrip("%").strip()
    text = str(duration or "").strip().lower()
    spelled = SPELLED_DURATION.match(text)
    if spelled:
        text = spelled.group(1) + spelled.group(2)[0]
    return build_command(name, turbine, percent=percent if name == "curtail" else None,
                         duration=text if name in ("curtail", "stop") else "", reason=reason,
                         requested_by=requested_by, request_id=request_id, now=now)


def find_item(items, name):
    """The id of the item of that display name in the answer of a Fabric list call, or None."""
    return next((item.get("id") for item in items if item.get("displayName") == name), None)


def find_custom_endpoint(topology, kind):
    """The id of the custom endpoint among the `sources` or the `destinations` of an eventstream, or None."""
    return next((node.get("id") for node in topology.get(kind, []) if node.get("type") == "CustomEndpoint"), None)


def send_settings(connection):
    """What a sender needs from the connection of a custom endpoint source, checked for completeness."""
    namespace = connection.get("fullyQualifiedNamespace") or ""
    event_hub = connection.get("eventHubName") or ""
    secret = (connection.get("accessKeys") or {}).get("primaryConnectionString") or ""
    missing = [label for label, value in (("fullyQualifiedNamespace", namespace), ("eventHubName", event_hub), ("accessKeys.primaryConnectionString", secret)) if not value]
    if missing:
        raise CommandError("The connection of the eventstream's source lacks: " + ", ".join(missing) + ".")
    return {"namespace": namespace, "eventHub": event_hub, "connectionString": secret}


def payload(command):
    """The bytes that go on the wire: compact JSON, as the simulator's own tool writes it."""
    return json.dumps(command, separators=(",", ":")).encode("utf-8")


import time
from datetime import datetime, timezone

TURBINES = [f"WT-{number:02d}" for number in range(1, 13)]
RATED_KW = 4200.0          # data/reference/turbines.csv; a test compares
WAIT_SECONDS = 150         # how long a command is waited for in the telemetry
POLL_SECONDS = 5
STOPPED = ("STOPPED", "MAINTENANCE")


def rows_of(body) -> list[dict]:
    """The rows of the first table of the KQL database's answer (REST, version 1), keyed by column name."""
    tables = (body or {}).get("Tables") if isinstance(body, dict) else None
    table = tables[0] if isinstance(tables, list) and tables else None
    if not isinstance(table, dict) or not isinstance(table.get("Columns"), list) or not isinstance(table.get("Rows"), list):
        raise RuntimeError("The KQL database answered in a form this notebook does not know (no Tables[0]).")
    names = [column.get("ColumnName") for column in table["Columns"]]
    rows = []
    for row in table["Rows"]:
        if not isinstance(row, list):       # a failure inside a result arrives as an object in place of a row
            raise RuntimeError(f"The KQL database reported a failure inside the result: {str(row)[:300]}")
        rows.append(dict(zip(names, row)))
    return rows


def targets_of(turbine: str) -> list[str]:
    """The turbines a command or a question is about. Refuses anything that is not a turbine or the word all."""
    name = str(turbine or "").strip()
    if name.lower() == "all":
        return list(TURBINES)
    if name.upper() not in TURBINES:
        raise ValueError(f"'{turbine}' is not a turbine. Use WT-01 to WT-12, or the word all.")
    return [name.upper()]


def _in_list(turbines: list[str]) -> str:
    return ", ".join(f"'{name}'" for name in turbines)


def status_query(turbine: str = "all") -> str:
    return f"""TurbineTelemetry
| where Timestamp > ago(10m) and TurbineId in ({_in_list(targets_of(turbine))})
| summarize arg_max(Timestamp, State, PowerSetpoint, ActivePower, WindSpeed, GearboxVibration, GearboxOilTemp) by TurbineId
| extend AgeSeconds = toint((now() - Timestamp) / 1s)
| order by TurbineId asc"""


def effect_query(turbine: str, since: str = "") -> str:
    """The newest reading of each turbine the command names: of the last five minutes, or, with `since`
    (the command's issuedAt), only readings taken after the command was written."""
    start = f"datetime({since})" if since else "ago(5m)"
    return f"""TurbineTelemetry
| where Timestamp >= {start} and TurbineId in ({_in_list(targets_of(turbine))})
| summarize arg_max(Timestamp, State, PowerSetpoint, ActivePower) by TurbineId
| extend AgeSeconds = toint((now() - Timestamp) / 1s)"""


def by_turbine(rows: list[dict]) -> dict:
    return {row.get("TurbineId"): row for row in rows}


def _number(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return number if number == number else None      # NaN is no number


def holds(cmd: dict, row) -> bool:
    """Does this reading show the command in force?"""
    if not row:
        return False
    state, setpoint = row.get("State"), _number(row.get("PowerSetpoint"))
    name = cmd["command"]
    if name == "curtail":
        return setpoint is not None and setpoint <= RATED_KW * float(cmd["percent"]) / 100.0 + 1.0
    if name == "release":
        return state != "CURTAILED"
    if name == "stop":
        return state in STOPPED
    if name == "start":
        return state not in STOPPED
    return state != "FAULT"                            # reset


def what(cmd: dict) -> str:
    name = cmd["command"]
    if name == "curtail":
        return f"limit of {RATED_KW * float(cmd['percent']) / 100.0:.0f} kW in force"
    return {"release": "limit lifted", "stop": "stopped", "start": "running again or free to run", "reset": "trip cleared"}[name]


def _reading(row) -> str:
    if not row:
        return "no reading"
    setpoint, power = _number(row.get("PowerSetpoint")), _number(row.get("ActivePower"))
    parts = [f"state {row.get('State')}"]
    if setpoint is not None:
        parts.append(f"setpoint {setpoint:.0f} kW")
    if power is not None:
        parts.append(f"power {power:.0f} kW")
    if row.get("Timestamp"):
        parts.append(f"reading of {str(row['Timestamp'])[11:19]}Z")
    return ", ".join(parts)


def follow(cmd: dict, before: dict, read, clock=time.monotonic, sleep=time.sleep, say=print,
           limit: float = WAIT_SECONDS, step: float = POLL_SECONDS) -> bool | None:
    """Wait until the telemetry shows the command, and say how long it took.

    `before` holds the newest reading of each turbine from before the send, `read()` returns the newest
    readings taken after it. A turbine whose reading already looked like the command's effect before
    the send proves nothing afterwards and is named as such, not counted as confirmed.
    Returns True when every other turbine showed the effect, False when the time ran out, and None
    when there was nothing the telemetry could show.
    """
    targets = targets_of(cmd["turbine"])
    already = [name for name in targets if holds(cmd, before.get(name))]
    waiting = [name for name in targets if name not in already]
    if already:
        say(f"{', '.join(already)}: already so before the command ({_reading(before.get(already[0]))}). "
            "For that the telemetry cannot show whether this command arrived.")
    if not waiting:
        return None
    say(f"Waiting for the telemetry to show it: {what(cmd)} on {', '.join(waiting)} (up to {limit:.0f} s) ...")
    began, latest = clock(), {}
    while True:
        latest = read() or {}
        for name in list(waiting):
            if holds(cmd, latest.get(name)):
                say(f"{name}: {what(cmd)}, seen {clock() - began:.0f} s after the send ({_reading(latest[name])}).")
                waiting.remove(name)
        if not waiting:
            return True
        if clock() - began >= limit:
            break
        sleep(step)
    for name in waiting:
        say(f"{name}: not seen after {limit:.0f} s ({_reading(latest.get(name) or before.get(name))}).")
    say("Follow the command hop by hop with the table 'If something fails' of docs/12-deployment-guide-phase5a.md: "
        "the eventstream's data preview first, then the control topic on the edge.")
    return False


def status_table(rows: list[dict]) -> str:
    """The state of the turbines as text, one line each."""
    if not rows:
        return ("No telemetry in the last 10 minutes. Is the edge VM running? It stops every day at 08:00 UTC: "
                "bash infra/vm.sh status")
    head = f"{'Turbine':<8} {'State':<17} {'Limit kW':>8} {'Power kW':>8} {'Wind m/s':>8} {'Vib mm/s':>8} {'Oil degC':>8} {'Age s':>6}"
    lines = [head, "-" * len(head)]

    def cell(value, digits, width):
        number = _number(value)
        return f"{'-':>{width}}" if number is None else f"{number:>{width}.{digits}f}"

    for row in rows:
        setpoint = _number(row.get("PowerSetpoint"))
        limit = f"{'-':>8}" if setpoint is None or setpoint >= RATED_KW - 1.0 else f"{setpoint:>8.0f}"
        lines.append(f"{str(row.get('TurbineId')):<8} {str(row.get('State')):<17} {limit} {cell(row.get('ActivePower'), 0, 8)} "
                     f"{cell(row.get('WindSpeed'), 1, 8)} {cell(row.get('GearboxVibration'), 2, 8)} {cell(row.get('GearboxOilTemp'), 1, 8)} "
                     f"{cell(row.get('AgeSeconds'), 0, 6)}")
    stamp = datetime.now(timezone.utc).strftime("%H:%M:%S")
    lines.append(f"{len(rows)} turbines, read at {stamp}Z. 'Limit' is the power setpoint when it is below the rated {RATED_KW:.0f} kW.")
    return "\n".join(lines)

In [ ]:
# Connect. Once per session: where to write commands, where to read the turbines' state.
import urllib.error
import urllib.request

import sempy.fabric as fabric

EVENTSTREAM = "WindFarmCommands"
KQL_DATABASE = "WindFarmEH"
client = fabric.FabricRestClient()
workspace_id = fabric.get_workspace_id()


def ask(path):
    response = client.get(path)
    if response.status_code != 200:
        raise SystemExit(f"The Fabric API answered {response.status_code} to {path}: {response.text[:300]}")
    return response.json()


eventstream_id = find_item(ask(f"v1/workspaces/{workspace_id}/eventstreams").get("value", []), EVENTSTREAM)
if not eventstream_id:
    raise SystemExit(f"No eventstream named {EVENTSTREAM} in this workspace. Run: bash infra/08-create-command-path.sh")
source_id = find_custom_endpoint(ask(f"v1/workspaces/{workspace_id}/eventstreams/{eventstream_id}/topology"), "sources")
if not source_id:
    raise SystemExit(f"The eventstream {EVENTSTREAM} has no custom endpoint as a source. Add one in the portal and publish.")
settings = send_settings(ask(f"v1/workspaces/{workspace_id}/eventstreams/{eventstream_id}/sources/{source_id}/connection"))
database = next((d for d in ask(f"v1/workspaces/{workspace_id}/kqlDatabases").get("value", []) if d.get("displayName") == KQL_DATABASE), None)
if database is None:
    raise SystemExit(f"No KQL database named {KQL_DATABASE} in this workspace.")
query_uri = database["properties"]["queryServiceUri"]
try:
    me = notebookutils.runtime.context.get("userName") or ""
except Exception:
    me = ""


def kql(query):
    """The rows of a query on the KQL database, as the signed-in user."""
    request = urllib.request.Request(
        f"{query_uri}/v1/rest/query", method="POST",
        data=json.dumps({"db": KQL_DATABASE, "csl": query}).encode("utf-8"),
        headers={"Authorization": f"Bearer {notebookutils.credentials.getToken(query_uri)}",
                 "Content-Type": "application/json; charset=utf-8", "Accept": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            return rows_of(json.loads(response.read()))
    except urllib.error.HTTPError as error:
        raise RuntimeError(f"The KQL database answered {error.code}: {error.read()[:300].decode('utf-8', 'replace')}") from None


def bring(module, pin):
    """Import a library, and install it first when this session does not have it.

    pip runs as a plain command here, not as the %pip magic: the magic is switched off when a notebook
    runs as a job, which is how the operations agent runs this one."""
    import importlib, site, subprocess, sys
    try:
        return importlib.import_module(module)
    except ImportError:
        done = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", pin],
                              capture_output=True, text=True)
        if done.returncode != 0:
            raise SystemExit(f"pip could not install {pin}: {(done.stderr or done.stdout)[-600:]}")
        if site.getusersitepackages() not in sys.path:   # where pip installs when the runtime's own folder is read-only
            sys.path.append(site.getusersitepackages())
        importlib.invalidate_caches()
        print(f"Installed {pin} for this session.")
        return importlib.import_module(module)


def status(turbine="all", say=print):
    """The state of the turbines, from the newest telemetry."""
    say(status_table(kql(status_query(turbine))))


def send(command, turbine, percent=None, duration="", reason="", wait=True, dry_run=False, say=print):
    """Build one command, check it, send it, and wait until the telemetry shows it.

    send("curtail", "WT-07", percent=30, duration="10m")     send("release", "WT-07")
    send("stop", "WT-05", duration="15m", reason="maintenance")     send("start", "WT-05")     send("reset", "WT-03")
    The command that was built last is kept in last_command."""
    global last_command
    try:
        cmd = from_parameters(command, turbine, percent=percent, duration=duration, reason=reason, requested_by=me)
    except CommandError as error:
        say(f"Not sent: {error}")
        return
    last_command = cmd
    say(json.dumps(cmd))
    if dry_run:
        say("dry_run: nothing was sent.")
        return
    before = by_turbine(kql(effect_query(cmd["turbine"]))) if wait else {}
    bring("websocket", "websocket-client==1.9.2")   # the transport over port 443; azure-eventhub does not bring it
    bring("azure.eventhub", "azure-eventhub==5.15.1")
    from azure.eventhub import EventData, EventHubProducerClient, TransportType

    extra = {} if "EntityPath=" in settings["connectionString"] else {"eventhub_name": settings["eventHub"]}
    producer = EventHubProducerClient.from_connection_string(
        settings["connectionString"], transport_type=TransportType.AmqpOverWebsocket, **extra)
    with producer:
        event = EventData(payload(cmd))
        event.content_type = "application/json"
        batch = producer.create_batch()
        batch.add(event)
        producer.send_batch(batch)
    say(f"Sent: {cmd['command']} for {cmd['turbine']}, request {cmd['requestId']}, at {cmd['issuedAt']} UTC.")
    if wait:
        follow(cmd, before, lambda: by_turbine(kql(effect_query(cmd["turbine"], since=cmd["issuedAt"]))), say=say)


last_command = None
print(f"Commands go to the eventstream {EVENTSTREAM} ({settings['namespace']}).")
print(f"The turbines' state is read from {KQL_DATABASE}." + (f" Commands are sent as {me}." if me else ""))
status()

## Where the turbines stand

Run this before and after anything. `status("WT-07")` shows one turbine.

In [ ]:
status()

## Limit a turbine, and lift the limit

`percent` is the share of its rated 4,200 kW the turbine may still produce; 30 is 1,260 kW. A curtailment needs a `duration` (`90s`, `10m`, `1h`) and ends by itself after it. In a weak wind the turbine produces less than the limit anyway: the state and the ring in the twin still change, the power does not.

In [ ]:
send("curtail", "WT-07", percent=30, duration="10m")

In [ ]:
send("release", "WT-07")

## Stop a turbine, and start it again

A stop with the reason `maintenance` shows as `MAINTENANCE`, with `operator` as `STOPPED`. With a `duration` the turbine starts again by itself; without one it stays down until `start`.

In [ ]:
send("stop", "WT-05", duration="5m", reason="maintenance")

In [ ]:
send("start", "WT-05")

## Reset a tripped turbine

A turbine that tripped on a fault shows `FAULT`. It starts again by itself once its alarms have cleared and a lockout time has passed; `reset` lifts the trip, and a derate the controller still holds, at once. On a turbine that has not tripped a reset changes nothing, and the notebook says so.

In [ ]:
send("reset", "WT-03")

## All twelve, or a look without sending

The word `all` in the place of the turbine means all twelve; nothing else does. `dry_run=True` builds and shows the command and sends nothing. `wait=False` sends and does not wait for the telemetry.

In [ ]:
send("curtail", "all", percent=50, duration="2m", dry_run=True)

## The same as a panel (optional)

Pick the values and select **Send**; **Status** shows the turbines. The box below the buttons is the log. If no panel appears, or a button does nothing, use the cells above: they do the same, and the panel adds nothing to them but the buttons.

In [ ]:
import ipywidgets as widgets

w_command = widgets.Dropdown(options=list(COMMANDS), value="curtail", description="Command")
w_turbine = widgets.Dropdown(options=TURBINES + ["all"], value="WT-07", description="Turbine")
w_percent = widgets.IntSlider(value=30, min=0, max=100, step=5, description="Percent")
w_duration = widgets.Text(value="10m", description="Duration")
w_reason = widgets.Dropdown(options=["operator", "maintenance"], value="operator", description="Stop as")
w_wait = widgets.Checkbox(value=True, description="Wait for the telemetry")
w_send = widgets.Button(description="Send", button_style="warning")
w_status = widgets.Button(description="Status")
w_log = widgets.Textarea(value="", disabled=True, layout=widgets.Layout(width="100%", height="260px"))


def _log(line):
    w_log.value = (w_log.value + str(line) + "\n")[-6000:]


def _run(action):
    w_log.value = ""
    try:
        action()
    except Exception as error:       # a button has nowhere else to show it
        _log(f"Failed: {error}")


w_send.on_click(lambda _: _run(lambda: send(
    w_command.value, w_turbine.value, percent=w_percent.value, duration=w_duration.value,
    reason=w_reason.value if w_command.value == "stop" else "", wait=w_wait.value, say=_log)))
w_status.on_click(lambda _: _run(lambda: status(say=_log)))
panel = widgets.VBox([widgets.HBox([w_command, w_turbine]), widgets.HBox([w_percent, w_duration, w_reason]),
                      widgets.HBox([w_send, w_status, w_wait]), w_log])
display(panel)